In [ ]:
# OpenCode / OpenAI-compatible notebook setup.
# Install once in the notebook's Python environment: pip install openai python-dotenv pyyaml
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os as _nb_os

from pathlib import Path as _NotebookPath

from dotenv import load_dotenv as _notebook_load_dotenv
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Paths
#
CHAPTER = "s05_todo_write"

PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if ((directory / CHAPTER / "code.py").is_file())
    ),
    None,
)

if (PROJECT_ROOT is None):

    raise (FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ "
        "so code.py can be located"
    ))
#

_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)

__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")
#
###################################################################################################
###################################################################################################
################################################################################################### Interactive Entry Point
#
# Run All must not start a blocking CLI. Enable this explicitly if needed.
RUN_AGENT_INTERACTIVELY = False
#
###################################################################################################
###################################################################################################
################################################################################################### OpenCode Configuration
#
# The existing ANTHROPIC_* environment names remain supported.
_notebook_key   = (_nb_os.getenv("OPENCODE_API_KEY")  or  _nb_os.getenv("ANTHROPIC_API_KEY"))
_notebook_model = (_nb_os.getenv("OPENCODE_MODEL")  or  _nb_os.getenv("MODEL_ID"))
_notebook_url   = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or  _nb_os.getenv("ANTHROPIC_BASE_URL")
    or  "https://opencode.ai/inference/openai/v1"
).rstrip("/")

if (_notebook_url.endswith("/chat/completions")):

    _notebook_url = _notebook_url[:-len("/chat/completions")]
#

if (not _notebook_key):

    raise (RuntimeError(
        "Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env"
    ))
#

if (not _notebook_model):

    raise (RuntimeError(
        "Set MODEL_ID (or OPENCODE_MODEL) in the root .env"
    ))
#

_nb_os.environ["OPENCODE_API_KEY"]   = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"]  = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"]          = _notebook_model

print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")
#


In [ ]:
# Bridge the lesson's Anthropic-style message interface to OpenCode's OpenAI API.
# Legacy chapter loaders import "anthropic" dynamically; this mapping serves those
# imports inside the current notebook kernel without changing the .py files.
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import json  as _compat_json
import sys   as _compat_sys
import types as _compat_types

from openai import OpenAI
import uuid
#
###################################################################################################
###################################################################################################
################################################################################################### Content Conversion
#
def _compat_field(block:object, key:str, default=None):

    """
        Read a response field from either a plain dictionary or provider object.

        The chapter code consumes both forms; this helper keeps those access
        patterns equivalent without mutating the original response.
    """

    return ((block.get(key, default))
            if (isinstance(block, dict))
            else (getattr(block, key, default)))
#


def _compat_text(value:object) -> str:

    """
        Convert Anthropic-style content blocks to Chat Completions text.

        Exclude tool-use blocks: their identifiers and arguments are translated
        separately so that subsequent tool results can be matched correctly.
    """

    if (isinstance(value, str)):

        return (value)
    #
    if (value is None):

        return ("")
    #
    if (isinstance(value, list)):

        return ("\n".join(
            _compat_text(_compat_field(item, "text", _compat_field(item, "content", "")))
            for item in value
            if (_compat_field(item, "type") != "tool_use")
        ))
    #
    if (isinstance(value, dict)):

        return (_compat_text(value.get(
            "text", value.get("content", _compat_json.dumps(value))
        )))
    #
    return (str(value))
#


def _compat_history(messages:list, system:str|list|None=None) -> list:

    """
        Translate chapter conversation history for OpenAI Chat Completions.

        Preserve every assistant tool-call ID and pair it with the corresponding
        tool response. The system prompt is inserted before conversational turns.
    """

    result = []

    if (system):

        result.append({"role": "system", "content": _compat_text(system)})
    #

    for message in messages:

        role    = _compat_field(message, "role")
        content = _compat_field(message, "content", "")

        if (role == "assistant"):

            if (not isinstance(content, list)):

                result.append({"role": "assistant", "content": _compat_text(content)})
                continue
            #

            text_blocks = []
            tool_calls  = []

            for block in content:

                kind = _compat_field(block, "type")

                if (kind == "tool_use"):

                    arguments = _compat_field(block, "input", {})

                    tool_calls.append({
                        "id"       : _compat_field(block, "id"),
                        "type"     : "function",
                        "function" : {
                            "name"      : _compat_field(block, "name"),
                            "arguments" : _compat_json.dumps(arguments, ensure_ascii=False),
                        },
                    })
                #
                elif (kind == "text"):

                    text_blocks.append(_compat_field(block, "text", ""))
                #
            #

            converted = {"role": "assistant", "content": "\n".join(text_blocks) or None}

            if (tool_calls):

                converted["tool_calls"] = tool_calls
            #
            if (converted["content"] is None  and  not tool_calls):

                converted["content"] = ""
            #

            result.append(converted)
        #
        elif (role == "user"  and  isinstance(content, list)):

            user_text    = []
            tool_results = []

            for block in content:

                if (_compat_field(block, "type") == "tool_result"):

                    tool_results.append({
                        "role"         : "tool",
                        "tool_call_id" : _compat_field(block, "tool_use_id"),
                        "content"      : _compat_text(_compat_field(block, "content", "")),
                    })
                #
                else:

                    user_text.append(_compat_text(block))
                #
            #

            result.extend(tool_results)

            if (user_text):

                result.append({"role": "user", "content": "\n".join(user_text)})
            #
        #
        else:

            result.append({"role": role, "content": _compat_text(content)})
        #
    #

    return (result)
#
###################################################################################################
###################################################################################################
################################################################################################### OpenAI Message Adapter
#
class _NotebookMessages:

    """
        Expose messages.create on an OpenAI-compatible chat-completions client.

        Construct function-tool definitions, convert responses to the chapter's
        expected block objects, and translate token counts and stop reasons.
    """

    def __init__(self, client) -> None:

        """
            Keep the provider client used by subsequent model requests.
        """

        self._client = client
    #
    #################################################################  Public API
    #
    def create(
            self,
            *,
            model      : str,
            messages   : list,
            system     : str|list|None = None,
            tools      : list|None     = None,
            max_tokens : int|None      = None,
            **kwargs,
        ) -> object:

        """
            Send an OpenAI-compatible request and return Anthropic-shaped blocks.

            Tool requests remain associated with their call identifiers.
            The translation preserves the original lesson's model/tool loop.
        """

        request = {"model": model, "messages": _compat_history(messages, system)}

        if (tools):

            request["tools"] = [
                {
                    "type"     : "function",
                    "function" : {
                        "name"        : tool["name"],
                        "description" : tool.get("description", ""),
                        "parameters"  : tool.get(
                            "input_schema", {"type": "object", "properties": {}}
                        ),
                    },
                }
                for tool in tools
            ]
        #
        if (max_tokens is not None):

            request["max_tokens"] = max_tokens
        #

        response = self._client.chat.completions.create(**request)
        choice   = response.choices[0]
        reply    = choice.message
        blocks   = []

        if (reply.content):

            blocks.append(_compat_types.SimpleNamespace(
                type = "text",
                text = reply.content,
            ))
        #

        for call in (reply.tool_calls or []):

            try:

                arguments = _compat_json.loads(call.function.arguments or "{}")

                if (not isinstance(arguments, dict)):

                    arguments = {}
                #
            #
            except (ValueError, TypeError):

                arguments = {}
            #

            blocks.append(_compat_types.SimpleNamespace(
                type  = "tool_use",
                id    = call.id,
                name  = call.function.name,
                input = arguments,
            ))
        #

        usage       = getattr(response, "usage", None)
        stop_reason = (
            ("max_tokens") if (choice.finish_reason == "length")
            else (("tool_use") if (reply.tool_calls) else ("end_turn"))
        )

        return (_compat_types.SimpleNamespace(
            content     = blocks,
            stop_reason = stop_reason,
            usage       = _compat_types.SimpleNamespace(
                input_tokens  = getattr(usage, "prompt_tokens", 0)  or  0,
                output_tokens = getattr(usage, "completion_tokens", 0)  or  0,
            ),
        ))
    #
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Client
#
class _NotebookAnthropic:

    """
        Provide a legacy chapter client backed by the OpenCode endpoint.

        The name is retained for import compatibility; requests are always sent
        through the OpenAI Python SDK and use the notebook's configured API key.
    """

    def __init__(self, base_url:str|None=None, **kwargs) -> None:

        """
            Initialize the OpenAI client and expose the message adapter.
        """

        url = (_nb_os.getenv("OPENCODE_BASE_URL")  or  base_url  or  _notebook_url)

        self._client = OpenAI(
            api_key         = _notebook_key,
            base_url        = url,
            default_headers = {
                "x-opencode-session" : f"ses_{uuid.uuid4().hex}",
                "User-Agent"        : "learn-claude-code/1.0",
            },
        )
        self.messages = _NotebookMessages(self._client)
    #
#


_anthropic_compat           = _compat_types.ModuleType("anthropic")
_anthropic_compat.Anthropic = _NotebookAnthropic
_compat_sys.modules["anthropic"] = _anthropic_compat
#


In [ ]:
#!/usr/bin/env python3
"""
s05_todo_write.py - TodoWrite

The model tracks its progress through a TodoManager. After three rounds
without an update, the harness adds a reminder alongside the tool results.

    +----------+      +-------+      +--------------+
    |   User   | ---> |  LLM  | ---> | Tools        |
    |  prompt  |      |       |      | + todo_write |
    +----------+      +---^---+      +------+-------+
                          |                 | update
                          |          +------v----------+
                          |          | TodoManager     |
                          |          | [ ] pending     |
                          |          | [>] in progress |
                          |          | [x] completed   |
                          |          +------+----------+
                          | tool_result     |
                          +-----------------+

              rounds_since_todo >= 3 -> add <reminder>
"""


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import ast
import json
import os
import re
import subprocess
from pathlib import Path

try:

    import readline
    readline.parse_and_bind('set bind-tty-special-chars off')
#
except ImportError:

    pass
#

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if (os.getenv("ANTHROPIC_BASE_URL")):

    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
#

WORKDIR = Path.cwd()
client  = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL   = os.environ["MODEL_ID"]

# s05 change: SYSTEM prompt adds planning guidance
ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax, and prefer dedicated file tools for file operations")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)
SYSTEM             = (
    f"You are a coding agent at {WORKDIR}. Environment: {ENVIRONMENT_PROMPT}. "
    "Before starting any multi-step task, use todo_write to plan your steps. "
    "Update status as you go."
)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### Tool implementations from s02-s04
#

def run_bash(command:str) -> str:

    try:

        r   = subprocess.run(
            command,
            shell          = True,
            cwd            = WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        out = (r.stdout + r.stderr).strip()
        return ((out[:50000]) if (out) else ("(no output)"))
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
#

def run_read(path:str, limit:int|None=None) -> str:

    try:

        lines = (WORKDIR / path).resolve().read_text(encoding="utf-8").splitlines()
        if (limit  and  limit < len(lines)):

            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        #
        return ("\n".join(lines))
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#

def run_write(path:str, content:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding="utf-8")
        return (f"Wrote {len(content)} bytes to {path}")
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#

def run_edit(path:str, old_text:str, new_text:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        text      = file_path.read_text(encoding="utf-8")
        if (old_text not in text):

            return (f"Error: text not found in {path}")
        #
        file_path.write_text(text.replace(old_text, new_text, 1), encoding="utf-8")
        return (f"Edited {path}")
    #
    except Exception as e:

        return (f"Error: {e}")
    #
#

def run_glob(pattern:str) -> str:

    import glob as g
    try:

        matches = sorted({
            match for match in g.glob(
            pattern,
            root_dir  = WORKDIR,
            recursive = True,
        )
            if (WORKDIR / match).resolve().is_relative_to(WORKDIR)
        })
        shown   = matches[:200]
        if (len(matches) > 200):

            shown.append("... (more matches omitted; narrow the pattern)")
        #
        return (("\n".join(shown)) if (shown) else ("(no matches)"))
    #
    except Exception as e:

        return (f"Error: {e}")


In [ ]:
    #
#


#
###################################################################################################
###################################################################################################
################################################################################################### New in s05: structured state the model updates
#

class TodoManager:

    #
    #################################################################  Inits
    #
    def __init__(self):

        self.items: list[dict] = []
    #

    #
    #################################################################  Public API
    #
    def update(self, todos:list|str) -> str:

        if (isinstance(todos, str)):

            try:

                todos = json.loads(todos)
            #
            except json.JSONDecodeError:

                try:

                    todos = ast.literal_eval(todos)
                #
                except (SyntaxError, ValueError) as e:

                    raise (ValueError("todos must be a list or JSON array string")) from e
                #
            #
        #

        if (not isinstance(todos, list)):

            raise (ValueError("todos must be a list"))
        #
        if (len(todos) > 20):

            raise (ValueError("Max 20 todos allowed"))
        #

        validated         = []
        in_progress_count = 0
        for index, todo in enumerate(todos):

            if (not isinstance(todo, dict)):

                raise (ValueError(f"todos[{index}] must be an object"))
            #

            content = str(todo.get("content", "")).strip()
            status  = str(todo.get("status", "pending")).lower()
            if (not content):

                raise (ValueError(f"todos[{index}] requires content"))
            #
            if (status not in ("pending", "in_progress", "completed")):

                raise (ValueError(f"todos[{index}] has invalid status '{status}'"))
            #
            if (status == "in_progress"):

                in_progress_count += 1
            #
            validated.append({"content": content, "status": status})
        #

        if (in_progress_count > 1):

            raise (ValueError("Only one todo can be in_progress at a time"))
        #

        self.items = validated
        return (self.render())
    #

    #
    #################################################################  Public API
    #
    def render(self) -> str:

        if (not self.items):

            return ("No todos.")
        #

        lines = []
        for todo in self.items:

            marker = {
                "pending"     : "[ ]",
                "in_progress" : "[>]",
                "completed"   : "[x]",
            }[todo["status"]]
            lines.append(f"{marker} {todo['content']}")
        #

        done = sum(todo["status"] == "completed" for todo in self.items)
        lines.append(f"\n({done}/{len(self.items)} completed)")
        return ("\n".join(lines))
    #
#


TODO = TodoManager()


def run_todo_write(todos:list|str) -> str:

    try:

        output = TODO.update(todos)
    #
    except ValueError as e:

        return (f"Error: {e}")
    #
    print(f"\n\033[33m## Current Tasks\033[0m\n{output}")
    return (output)
#

TOOLS = [
    {
        "name"         : "bash",
        "description"  : "Run a shell command.",
        "input_schema" : {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    },
    {
        "name"         : "read_file",
        "description"  : "Read file contents.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    },
    {
        "name"         : "write_file",
        "description"  : "Write content to a file.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    },
    {
        "name"         : "edit_file",
        "description"  : "Replace exact text in a file once.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    },
    {
        "name"         : "glob",
        "description"  : "Find files matching a glob pattern; ** matches recursively.",
        "input_schema" : {"type": "object", "properties": {"pattern": {"type": "string"}}, "required": ["pattern"]},
    },
    # s05: new tool
    {
        "name"         : "todo_write",
        "description"  : "Create and manage a task list for your current coding session.",
        "input_schema" : {"type": "object", "properties": {"todos": {"type": "array", "maxItems": 20, "items": {"type": "object", "properties": {"content": {"type": "string", "minLength": 1}, "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]}}, "required": ["content", "status"]}}}, "required": ["todos"]},
    },
]

TOOL_HANDLERS = {
    "bash"       : run_bash,
    "read_file"  : run_read,
    "write_file" : run_write,
    "edit_file"  : run_edit,
    "glob"       : run_glob,
    "todo_write" : run_todo_write,
}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### Hook system from s04
#

HOOKS = {"UserPromptSubmit": [], "PreToolUse": [], "PostToolUse": [], "Stop": []}

def register_hook(event:str, callback):

    HOOKS[event].append(callback)
#

def trigger_hooks(event:str, *args):

    for callback in HOOKS[event]:

        result = callback(*args)
        if (result is not None):

            return (result)
        #
    #
    return (None)
#

DENY_LIST                = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if="]
DESTRUCTIVE_COMMAND_WORD = re.compile(
    r"(?i)(?:^|[;&|()\n])\s*(?:rm|del)(?=\s|$|[;&|()])"
)
DESTRUCTIVE              = ["rm ", "> /etc/", "chmod 777"]


def contains_destructive_command(command:str) -> bool:

    return (bool(DESTRUCTIVE_COMMAND_WORD.search(command)))
#


def permission_hook(block):

    """PreToolUse: s03 permission logic, registered as an s04 hook."""
    if (block.name == "bash"):

        command = block.input.get("command", "")
        for pattern in DENY_LIST:

            if (pattern in command):

                print(f"\n\033[31m[blocked] '{pattern}'\033[0m")
                return ("Permission denied by deny list")
            #
        #
        if (contains_destructive_command(command)  or  any(
            keyword in command for keyword in DESTRUCTIVE
        )):

            print(f"\n\033[33m[permission] Potentially destructive command\033[0m")
            print(f"   Tool: {block.name}({block.input})")
            choice = input("   Allow? [y/N] ").strip().lower()
            if (choice not in ("y", "yes")):

                return ("Permission denied by user")
            #
        #
    #
    if (block.name in ("read_file", "write_file", "edit_file")):

        path = block.input.get("path", "")
        if (not (WORKDIR / path).resolve().is_relative_to(WORKDIR)):

            print(f"\n\033[33m[permission] Access outside workspace\033[0m")
            print(f"   Tool: {block.name}({block.input})")
            choice = input("   Allow? [y/N] ").strip().lower()
            if (choice not in ("y", "yes")):

                return ("Permission denied by user")
            #
        #
    #
    return (None)
#

def log_hook(block):

    """PreToolUse: log every tool call."""
    args_preview = str(list(block.input.values())[:2])[:60]
    print(f"\033[90m[HOOK] {block.name}({args_preview})\033[0m")
    return (None)
#

def large_output_hook(block, output):

    """PostToolUse: warn on large output."""
    if (len(str(output)) > 100000):

        print(f"\033[33m[HOOK] Large output from {block.name}: {len(str(output))} chars\033[0m")
    #
    return (None)
#

def context_inject_hook(query:str):

    """UserPromptSubmit: log working directory."""
    print(f"\033[90m[HOOK] UserPromptSubmit: working in {WORKDIR}\033[0m")
    return (None)
#

def summary_hook(messages:list):

    """Stop: print tool call count."""
    tool_count = sum(1 for m in messages
                     for b in ((m.get("content")) if (isinstance(m.get("content"), list)) else ([]))
                     if isinstance(b, dict)  and  b.get("type") == "tool_result")
    print(f"\033[90m[HOOK] Stop: session used {tool_count} tool calls\033[0m")
    return (None)
#

register_hook("UserPromptSubmit", context_inject_hook)
register_hook("PreToolUse", permission_hook)
register_hook("PreToolUse", log_hook)
register_hook("PostToolUse", large_output_hook)
register_hook("Stop", summary_hook)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### Agent loop with the reminder counter
#

def agent_loop(messages:list):

    rounds_since_todo = 0
    while (True):

        response = client.messages.create(
            model      = MODEL,
            system     = SYSTEM,
            messages   = messages,
            tools      = TOOLS,
            max_tokens = 8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (not tool_calls):

            force = trigger_hooks("Stop", messages)
            if (force):

                messages.append({"role": "user", "content": force})
                continue
            #
            return
        #

        results   = []
        used_todo = False
        for block in tool_calls:

            blocked = trigger_hooks("PreToolUse", block)
            if (blocked):

                results.append({
                    "type"        : "tool_result",
                    "tool_use_id" : block.id,
                    "content"     : str(blocked),
                })
                continue
            #

            handler = TOOL_HANDLERS.get(block.name)
            try:

                output = (handler(**block.input)) if (handler) else (f"Unknown: {block.name}")
            #
            except Exception as e:

                output = f"Error: {e}"
            #

            trigger_hooks("PostToolUse", block, output)

            if (block.name == "todo_write"):

                used_todo = True
            #

            results.append({
                "type"        : "tool_result",
                "tool_use_id" : block.id,
                "content"     : str(output),
            })
        #

        rounds_since_todo = (0) if (used_todo) else (rounds_since_todo + 1)
        if (rounds_since_todo >= 3):

            results.append({
                "type" : "text",
                "text" : "<reminder>Update your todos.</reminder>",
            })
            rounds_since_todo = 0
        #

        messages.append({"role": "user", "content": results})
    #
#


if (RUN_AGENT_INTERACTIVELY):

    print("s05: TodoWrite - plan before execution")
    print("Enter a question, press Enter to send. Type q to quit.\n")

    history = []
    while (True):

        try:

            # \001/\002 tell Readline the ANSI escapes have zero display width.
            query = input("\001\033[36m\002s05 >> \001\033[0m\002")
        #
        except (EOFError, KeyboardInterrupt):

            break
        #
        if (query.strip().lower() in ("q", "exit", "")):

            break
        #
        trigger_hooks("UserPromptSubmit", query)
        history.append({"role": "user", "content": query})
        agent_loop(history)
        for block in history[-1]["content"]:

            if (getattr(block, "type", None) == "text"):

                print(block.text)
            #
        #
        print()
    #
#
